# Advanced Reactor Design: Non-Ideal Flow and Residence Time Distribution (RTD)

**Objective:** This lesson moves beyond ideal reactor models to the real world. We will explore how to characterize non-ideal flow in reactors using Residence Time Distribution (RTD) and then use this information to predict the reactor's actual performance.

**Learning Goals:**
1.  Understand the concepts of non-ideal flow, including dead zones and bypassing.
2.  Learn how a tracer experiment is used to generate an RTD curve ($E(t)$).
3.  Model a non-ideal reactor using the **Tanks-in-Series (TIS)** model, a powerful tool for representing intermediate mixing.
4.  Use the **Segregated Flow Model** to predict the conversion of a chemical reaction in a non-ideal reactor, bridging the gap between RTD data and performance prediction.

## Part 1: The Reality of Non-Ideal Flow

Ideal CSTR and PFR models are the cornerstones of reactor design, but real reactors are never perfect. They suffer from complex flow patterns:
*   **Dead Zones:** Regions of the reactor with low fluid turnover, effectively reducing the active volume of the reactor.
*   **Bypassing or Channeling:** When a portion of the fluid takes a shortcut from the inlet to the outlet, leading to a very short residence time and low conversion.

The **Residence Time Distribution (RTD)** is the experimental tool we use to diagnose these problems. It's a probability distribution that tells us what fraction of the fluid exiting the reactor spent a certain amount of time inside it. We measure it by injecting a pulse of an inert tracer at the inlet and measuring its concentration at the outlet over time.

## Part 2: Modeling Non-Ideality - The Tanks-in-Series (TIS) Model

One of the most effective ways to model a non-ideal reactor is to represent it as a cascade of $N$ equal-sized ideal CSTRs connected in series. This model is incredibly versatile:

*   If $N=1$, the model represents a **perfect CSTR**.
*   If $N \rightarrow \infty$, the model represents a **perfect PFR**.
*   For intermediate values of $N$, the model represents a reactor with a degree of dispersion or back-mixing that lies between the two ideals. The value of $N$ can be found by fitting the model to experimental RTD data.

The E-curve (the RTD function) for the TIS model is given by:
$$ E(t) = \frac{N}{\tau} \frac{(Nt/\tau)^{N-1}}{(N-1)!} e^{-Nt/\tau} $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import gamma # Gamma(N) = (N-1)!

# --- System Parameters ---
tau = 10.0      # Overall mean residence time (minutes)
t = np.linspace(0, 4*tau, 500)

def e_curve_tis(t, tau, N):
    """E-Curve for the Tanks-in-Series model."""
    # Handle t=0 case to avoid log(0) issues
    t_safe = np.maximum(t, 1e-9)
    return (N/tau) * (N*t_safe/tau)**(N-1) / gamma(N) * np.exp(-N*t_safe/tau)

# --- Plotting E-Curves for different N ---
plt.figure(figsize=(12, 7))
N_values = [1, 2, 5, 10, 50]
for N in N_values:
    E_tis = e_curve_tis(t, tau, N)
    plt.plot(t, E_tis, label=f'Tanks-in-Series, N={N}')

plt.title('Tanks-in-Series RTD Model: Bridging CSTR and PFR', fontsize=16, weight='bold')
plt.xlabel('Time, t (minutes)', fontsize=14)
plt.ylabel('E(t) (1/min)', fontsize=14)
plt.legend()
plt.show()

print("Observe how as N increases, the distribution becomes narrower and more symmetric. For N=1, we have the classic CSTR exponential decay. As N gets large, the curve approaches a sharp pulse, characteristic of a PFR.")

## Part 3: Predicting Conversion with the Segregated Flow Model

The real power of RTD comes from its ability to predict reactor performance. The **Segregated Flow Model** is a powerful way to do this.

**The Concept:** We imagine the fluid flowing through our non-ideal reactor not as a continuum, but as a vast collection of tiny, separate fluid 'packets'. Each packet acts as its own tiny batch reactor. The final outlet stream is simply the mixture of all these packets, each of which has spent a different amount of time reacting.

The overall average conversion ($X_A$) is therefore the average of the conversions achieved in each packet, weighted by the RTD function, $E(t)$:
$$ \bar{X}_A = \int_0^\infty X_{A,batch}(t) \cdot E(t) \, dt $$
where $X_{A,batch}(t)$ is the conversion achieved in an ideal batch reactor after time $t$.

In [ ]:
# --- Define the reaction and batch kinetics ---
k = 0.25 # Rate constant for a first-order reaction (1/min)

def batch_conversion(t, k):
    """Conversion for a first-order reaction in a batch reactor."""
    # X_A = 1 - C_A/C_A0 = 1 - exp(-kt)
    return 1 - np.exp(-k * t)

# --- Calculate Conversion for Reactors of a given tau ---

# 1. Ideal PFR: All fluid spends exactly tau time
conversion_pfr = batch_conversion(tau, k)

# 2. Ideal CSTR: Has a known analytical solution
conversion_cstr = (k * tau) / (1 + k * tau)

# 3. Non-Ideal TIS Reactor: We must perform the integration
def calculate_non_ideal_conversion(N, tau, k):
    """Calculates conversion for a TIS reactor using the Segregated Flow Model."""
    # The function we need to integrate
    def integrand(t, k, tau, N):
        return batch_conversion(t, k) * e_curve_tis(t, tau, N)
    
    # Perform the numerical integration from t=0 to t=infinity
    conversion, error = quad(integrand, 0, np.inf, args=(k, tau, N))
    return conversion

# --- Run the calculation for a range of N values ---
N_range = np.arange(1, 31)
conversions_non_ideal = [calculate_non_ideal_conversion(N, tau, k) for N in N_range]

print("Calculations complete.")

In [ ]:
# --- Plotting the Results ---
plt.figure(figsize=(12, 7))

plt.plot(N_range, conversions_non_ideal, 'o-', label='Non-Ideal Reactor (TIS Model)')
plt.axhline(y=conversion_pfr, color='blue', linestyle='--', label=f'Ideal PFR Limit (X={conversion_pfr:.3f})')
plt.axhline(y=conversion_cstr, color='red', linestyle='--', label=f'Ideal CSTR Limit (X={conversion_cstr:.3f})')

plt.title('Effect of Non-Ideal Mixing on Reactor Conversion', fontsize=16, weight='bold')
plt.xlabel('Number of Tanks in Series, N (Degree of Mixing)', fontsize=14)
plt.ylabel('Predicted Conversion, $X_A$', fontsize=14)
plt.legend()
plt.xscale('log') # Use a log scale for N to better see the behavior
plt.grid(True, which='both')
plt.show()

print("This plot is the key takeaway: a real reactor's performance lies on a spectrum. At N=1, our model correctly predicts the CSTR conversion. As N increases (less back-mixing), the performance steadily improves and approaches the ideal PFR limit.")

## Student Challenges

1.  **Second-Order Reaction:** For non-linear kinetics, the effect of the RTD is even more pronounced. Repeat the final analysis, but this time for a **second-order reaction**. You will need to change the `batch_conversion` function to: $X_A(t) = \frac{k C_{A0} t}{1 + k C_{A0} t}$. Is the gap in performance between the CSTR and PFR limits larger or smaller than for the first-order case? Why?

2.  **Diagnosing a Real Reactor:** An experimental RTD curve for a real reactor gives a mean residence time of $\tau=15$ min and a variance of $\sigma^2 = 75$ min$^2$. For the Tanks-in-Series model, the relationship is $\sigma^2 = \frac{\tau^2}{N}$. Calculate the value of $N$ that best describes this real reactor. Using this $N$, predict the conversion for the first-order reaction (k=0.25$ min^{-1}). How does it compare to the ideal reactors with the same $\tau$?